<a href="https://colab.research.google.com/github/dralexlup/Jarvis/blob/master/LLM_Response_testing_Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title # Cell 0: Setup, Dependencies, and Authentication (Corrected)
# @markdown This cell installs necessary libraries and handles authentication for both
# @markdown Google Drive and the Google Docs API.

# --- Install required packages ---
print("Installing required packages...")
!pip install -q --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install -q transformers torch accelerate bitsandbytes sentencepiece tqdm
print("✅ Installation complete.")

# --- Import necessary libraries ---
import os
import random
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from google.colab import drive, auth
from googleapiclient.discovery import build
from tqdm.auto import tqdm
from IPython.display import clear_output
print("✅ Imports complete.")

# --- Authenticate and Mount Google Drive ---
print("\nMounting Google Drive...")
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive mounted successfully at /content/drive.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")

# --- Authenticate for Google APIs (Docs & Drive) ---
# FIX: The new, correct way to authenticate in Colab.
# This single call handles the user authentication pop-up.
# The 'build' function in a later cell will automatically use these credentials.
print("\nAuthenticating for Google APIs...")
try:
    auth.authenticate_user()
    print("✅ Google API authentication successful. Credentials are now available for this session.")
except Exception as e:
    print(f"❌ Error during authentication: {e}")

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 36.2 MB/s eta 0:00:00
✅ Installation complete.
✅ Imports complete.

Mounting Google Drive...
Mounted at /cont

In [ ]:
# @title #HuggingFace Login
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The token `Main` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate whe

In [ ]:
# @title # Cell 1: Load Model & Generate Responses (Enhanced & Fixed)
# @markdown ---
# @markdown ### 1. Enter the Hugging Face Model ID
# @markdown Paste the full Hub ID of the model you want to test (e.g., `google/gemma-2-9b-it`, or your abliterated model `your-username/DrMedra4B-abliterated`).
HUB_REPO_ID = "drwlf/Medra4b" # @param {type:"string"}

# @markdown ---
# @markdown ### 2. Define Prompts
# @markdown Provide a system message to define the model's persona and a user prompt to generate responses for.
system_message = "You are Medra, an advanced AI medical assistant. You are trained to provide compassionate, factual, and comprehensive medical information to both professionals and non-experts. You must provide your reasoning within <think> tags before your final answer." # @param {type:"string"}
user_prompt = "Cum decid care este cea mai buna varianta chirurgicala pentru a opera un cleft lip la un copil de 3 ani? Descrie-mi operatia pas cu pas." # @param {type:"string"}

# @markdown ---
# @markdown ### 3. Generation Settings
# @markdown Set the number of responses, token limit, and the repetition penalty. A penalty > 1.0 discourages repeating tokens.
number_of_responses = 20 # @param {type:"integer"}
max_new_tokens = 1024 # @param {type:"integer"}
repetition_penalty = 1.1 # @param {type:"slider", min:1.0, max:1.5, step:0.01}
# @markdown ---
import numpy as np
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from tqdm.auto import tqdm
from IPython.display import clear_output

# Clear previous output for a clean run
clear_output(wait=True)

# This list will store all the generated answers for saving to the Google Doc in Cell 2
generated_responses = []

# --- Set PyTorch Matmul Precision for better performance on compatible GPUs ---
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    print("Setting torch.set_float32_matmul_precision('high')")
    torch.set_float32_matmul_precision('high')

# --- Check if model is already loaded and matches the requested ID ---
# We store the loaded model's name in a global state variable to avoid reloading.
if 'loaded_model_id' not in globals() or globals().get('loaded_model_id') != HUB_REPO_ID:
    print(f"Loading model and tokenizer from: {HUB_REPO_ID}...")

    # Clear memory from any previously loaded model
    if 'model' in globals():
        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    try:
        model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

        model = AutoModelForCausalLM.from_pretrained(
            HUB_REPO_ID,
            trust_remote_code=True,
            torch_dtype=model_dtype,
            device_map="auto"
        ).eval()

        tokenizer = AutoTokenizer.from_pretrained(
            HUB_REPO_ID,
            trust_remote_code=True
        )

        globals()['loaded_model_id'] = HUB_REPO_ID # Store the ID of the successfully loaded model
        print("✅ Model and tokenizer loaded successfully.")
    except Exception as e:
        print(f"❌ An error occurred during model loading: {e}")
        model = None
else:
    print(f"✅ Model '{HUB_REPO_ID}' is already loaded. Skipping reload.")
    # Ensure model and tokenizer are pulled from the global scope if they exist
    model = globals().get('model')
    tokenizer = globals().get('tokenizer')


# --- Proceed only if model was loaded successfully ---
if 'model' in globals() and model is not None and 'tokenizer' in globals() and tokenizer is not None:
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"\nGenerating {number_of_responses} different responses...")
    print("-" * 70)
    print(f"SYSTEM: {system_message}")
    print(f"USER: {user_prompt}")
    print(f"REPETITION PENALTY: {repetition_penalty}")
    print("-" * 70)

    # --- Prepare Inputs ---
    conversation = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    # Robustly handle tokenizer output (dict or tensor)
    inputs_data = tokenizer.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if isinstance(inputs_data, torch.Tensor):
        inputs = {'input_ids': inputs_data.to(model.device)}
        inputs['attention_mask'] = torch.ones_like(inputs['input_ids'])
    elif isinstance(inputs_data, dict):
        inputs = {k: v.to(model.device) for k, v in inputs_data.items()}
    else:
        raise TypeError(f"Tokenizer returned an unexpected type: {type(inputs_data)}")


    # --- Generate Multiple Responses ---
    for i in tqdm(range(number_of_responses), desc="Generating Responses"):
        temp = round(random.uniform(0.6, 1.1), 2)
        top_p = round(random.uniform(0.9, 1.0), 2)

        print(f"\n--- Response {i+1}/{number_of_responses} (Temp: {temp}, Top-p: {top_p}) ---")

        # The streamer will print the output token by token in real-time
        streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                streamer=streamer,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temp,
                top_p=top_p,
                repetition_penalty=repetition_penalty, # <-- FIX: Added repetition_penalty parameter
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the full response separately to store it for the Google Doc
        response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        generated_responses.append(response_text.strip())

        print() # Add a newline for clean separation after streamed output

        # Clean up memory
        del outputs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n" + "-" * 70)
    print(f"✅ Finished generating {number_of_responses} responses. They are now stored and ready to be saved in Cell 2.")



Generating Responses:   0%|          | 0/20 [00:00<?, ?it/s]


--- Response 1/20 (Temp: 1.02, Top-p: 0.99) ---
<think>
Alright, so we have a young patient with a cleft lip in need of surgery at just three years old. I want it out there that this isn't one size fits all when dealing with clefts. Every kid is unique—not just on looks, but also how their bone structure or soft tissues developed can influence what kind of surgical decisions best suit them. It feels important to make sure any suggestions here line up with the actual anatomy seen by the pediatric surgeon on the day of operation. This avoids any potential overkill procedures as well, which can lead us down some tricky paths later on if things don't align perfectly from the start. So yeah, always keep context right where it belongs, with the kid, specifically!
</think>

When deciding who needs surgical correction for a cleft lip, doctors need to consider various factors specific to each child's situation. As they say, every person is different; not only do individuals develop differently

In [ ]:
# @title # Cell 2: Save Responses to Google Docs (Corrected)
# @markdown This cell will create a new Google Document in the `AI Work` folder on your
# @markdown Google Drive and write all the generated responses into it.

# --- Configuration ---
# @markdown The name of the folder inside "My Drive" to save the document.
# @markdown If this folder doesn't exist, it will be created.
output_folder_name = "AI Work" # @param {type:"string"}
# ---

# Check if responses were generated in the previous cell
if 'generated_responses' not in locals() or not generated_responses:
    print("❌ No responses found. Please run Cell 1 to generate responses first.")
else:
    print("--- Preparing to create Google Doc ---")

    # 1. Build the Google Docs and Drive API services
    try:
        # The 'build' function will automatically find the credentials
        # established by 'auth.authenticate_user()' in Cell 0.
        docs_service = build('docs', 'v1')
        drive_service = build('drive', 'v3')
        print("✅ Google Docs and Drive services initialized.")
    except Exception as e:
        print(f"❌ Failed to build Google services. Did you successfully authenticate in Cell 0? Error: {e}")
        docs_service = None

    if docs_service and drive_service:
        # 2. Find or create the target folder in Google Drive
        folder_id = None
        try:
            # Search for the folder by name in "My Drive"
            query = f"name='{output_folder_name}' and mimeType='application/vnd.google-apps.folder' and 'root' in parents and trashed=false"
            response = drive_service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
            files = response.get('files', [])

            if files:
                folder_id = files[0].get('id')
                print(f"Found existing folder '{output_folder_name}' with ID: {folder_id}")
            else:
                # Create the folder if it doesn't exist
                print(f"Folder '{output_folder_name}' not found. Creating it...")
                file_metadata = {
                    'name': output_folder_name,
                    'mimeType': 'application/vnd.google-apps.folder'
                }
                folder = drive_service.files().create(body=file_metadata, fields='id').execute()
                folder_id = folder.get('id')
                print(f"✅ Created folder '{output_folder_name}' with ID: {folder_id}")

        except Exception as e:
            print(f"❌ Error finding or creating Google Drive folder: {e}")
            folder_id = None # Set to None so it creates in root as a fallback

        # 3. Construct the document title and content
        sanitized_model_id = HUB_REPO_ID.replace("/", "_")
        doc_title = f"Responses from {sanitized_model_id} - {user_prompt[:30].strip()}..."

        content = f"Model: {HUB_REPO_ID}\n\n"
        content += f"System Message:\n{system_message}\n\n"
        content += f"User Prompt:\n{user_prompt}\n\n"
        content += "=" * 50 + "\n\n"

        for i, response in enumerate(generated_responses):
            content += f"--- Response {i+1}/{len(generated_responses)} ---\n"
            content += response
            content += "\n\n"

        # 4. Create the Google Doc and insert the text
        try:
            # --- FIX: Use the Drive API to create the file in the correct folder ---
            print(f"Creating Google Doc titled: '{doc_title}'...")
            file_metadata = {
                'name': doc_title,
                'mimeType': 'application/vnd.google-apps.document'
            }
            # Specify parent folder if it was found/created
            if folder_id:
                file_metadata['parents'] = [folder_id]

            # Use the Drive API's files().create method
            doc = drive_service.files().create(body=file_metadata, fields='id').execute()
            doc_id = doc.get('id')
            doc_link = f"https://docs.google.com/document/d/{doc_id}/edit"
            print(f"✅ Document created successfully. Link: {doc_link}")
            # --- End of FIX ---

            print("Inserting content into the document...")
            requests = [
                {
                    'insertText': {
                        'location': {
                            'index': 1,
                        },
                        'text': content
                    }
                }
            ]
            # Now use the Docs API with the created doc_id to add content
            docs_service.documents().batchUpdate(documentId=doc_id, body={'requests': requests}).execute()
            print("✅ All responses have been written to the Google Doc.")

        except Exception as e:
            print(f"❌ An error occurred while creating or writing to the Google Doc: {e}")



--- Preparing to create Google Doc ---
✅ Google Docs and Drive services initialized.
Found existing folder 'AI Work' with ID: 1wuriC0MxXAzjWmOxT-4niPmV8em0W7m9
Creating Google Doc titled: 'Responses from drwlf_Medra4b - Care e mecanismul imuno patoge...'...
✅ Document created successfully. Link: https://docs.google.com/document/d/1RG-dqGt_mkxSDLGF2_AGzoRh7z8e389u_4CiYp7HG4Q/edit
Inserting content into the document...
✅ All responses have been written to the Google Doc.
